### Import Dependencies

In [18]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams,Distance,PayloadSchemaType,PointStruct,SparseVectorParams,Document,Prefetch,FusionQuery
from qdrant_client import models
import pandas as pd
from openai import OpenAI
import fastembed
import os
import cohere
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
client=OpenAI()

In [3]:
qdrant_client=QdrantClient(url="http://localhost:6333")


In [8]:
def get_embedding(text,models="text-embedding-3-small"):
    response=client.embeddings.create(
        input=[text],
        model=models
    )
    return response.data[0].embedding

In [9]:
def reteriver_data(query, quadrant_client, k):
    query_embedding = get_embedding(query)

    result = quadrant_client.query_points(
        collection_name="Amazon-items-collection-02-hybrid-serach",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-model-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )

        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


In [54]:
query="give me some good laptop"
result=reteriver_data(query=query,quadrant_client=qdrant_client,k=10)

In [55]:
result[0]

["145Pcs Water Bottle Stickers for Girls Teens Kids Waterproof Inspirational VSCO Vinyl Stickers for Laptop Skateboard Aesthetic Trendy Cute Word Stickers School Reward Decals⭐ Girls Water Bottle Stickers - Total 145pcs cute motivational stickers for girl teens, not repeat, as picture show, HD image and kids friendly cute positive words VSCO pattern⭐ Unique Design & Material - kids water bottle stickers are waterproof and anti-sun, reusable, remove without residue and non-toxic , easy to peel off and stick, good sizes, about 1.5 inch to 4.3 inch⭐ Wide Application - Postive stickers fit for water bottle, laptop, skateboard, envelope, scrapbook, phone case, suitcase, helmet, guitar, cabinet, table, door, window, luggage, car, mirror, candy bag etc. Pasting these art stickers in different places of your life can inspire you anytime or anywhere⭐ Vinyl Stickers Pack - Waterproof stickers with loverly motivational word, can be given to all ages including teens, kids, adults, girls, teachers 

### Reranking using Cohere Api

In [56]:
CO_API_KEY=os.getenv("CO_API_KEY")

In [57]:
cohere_client=cohere.ClientV2(api_key=CO_API_KEY)

In [58]:
to_rerank=result[0]

In [62]:
response_rerank=cohere_client.rerank(
    model= "rerank-v4.0-fast",##"rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=10
)


In [63]:
response_rerank.results[0].index

4

In [64]:
for i in range(0,len(response_rerank.results)):
    print(f"{i} -> {to_rerank[response_rerank.results[i].index]}")


0 -> HP 2022 Newest Pavilion 15.6" FHD 1080P IPS Laptop, 8-Core AMD Ryzen 7-5700U(Up to 4.3GHz, Beat i7-1180G7), 32GB RAM, 1TB NVMe SSD, Numpad, HDMI, WiFi, USB-A&C, Fast Charge, Audio by B&O, Win11【15.6" FHD IPS micro-edge Display】Ultra-wide 178-degree viewing angles with consistent detail and a vibrant 1920 x 1080 resolution, you'll always have a great view of your favorite content.【AMD Ryzen 7 5700U】Push the limit for desktop computing with AMD Ryzen 7 processors. Their bold new architecture delivers intuitive cooling and multi-core processing for gaming, streaming, creativity, VR, and more.【Upgrade to 32GB DDR4 RAM】Get the mighty performance out of your laptop with support of the latest DDR4-3200 memory. Enjoy the faster system speed and responsiveness, the new standard will take your gaming experience to the next level.【Upgraded storage to 1TB PCIe SSD】Provides massive storage space for huge files, so that you can store important digital data and work your way through it with ease